# OSRT v6 — GRPO on Colab (G4 / RTX PRO 6000 Blackwell, 96GB)

RL with verifiable maths rewards, starting from the SFT-v4 checkpoint soup (**17.5% GSM8K reasoning-on vs 12.0% off**).

**Runtime: G4.** 96GB — more than an H100 — and sm_120 is verified end-to-end for this model. midtrain3 measured ~6,300 tok/s compiled, about 63% of H100 (this model is compute-bound, so GDDR7's lower bandwidth barely bites).

**Secrets:** add `HF_TOKEN` and `WANDB_API_KEY` in the Colab key sidebar (🔑).

**Why this exists rather than the Modal stage:** the Modal GRPO loop generates one prompt at a time (batch 16) and computes log-probs one sequence at a time (batch 1, twice per rollout). Measured **~5.5 min/step** — 900 steps would be 75 hours. `osrt.grpo_train` batches both. Expect **~60s/step** on G4, so 900 steps ≈ 15 hours.

---
### The four Colab fixes (each cost a debugging round on midtrain3 — do not skip)
1. **`--auth=adc`** on every `colab` CLI call. oauth2 silently drops the `colaboratory` scope on refresh → keep-alive 403s → VM reclaimed mid-run.
2. **No DataLoader workers.** Spawned workers hit a fatal `PyGILState_Release` teardown race (tokenizers/pyarrow + torch). This script uses none.
3. **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** — set by the script itself.
4. **Read progress from W&B / HF, not `colab exec`** — its websocket reports a false "step 0" when the run is much further along.

In [ ]:
# ── 1. Repo + deps + secrets ─────────────────────────────────────────
import os, sys

BRANCH = "feat/sft-harvest"
if not os.path.isdir("/content/osrt"):
    !git clone -q --depth 1 -b {BRANCH} https://github.com/CodeHalwell/OSRT-605M-A269M.git /content/osrt
%pip -q install -U "transformers>=5.3.0" datasets tokenizers safetensors wandb huggingface_hub
sys.path.insert(0, "/content/osrt/src")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
cap = torch.cuda.get_device_capability(0)
print(torch.cuda.get_device_name(0), f"| sm_{cap[0]}{cap[1]} |",
      f"{torch.cuda.get_device_properties(0).total_memory/2**30:.0f}GB | torch", torch.__version__)

In [ ]:
# ── 2. Prompt set: UNSEEN problems with numeric gold ─────────────────
# Must be problems SFT-v4 never trained on. Its builder consumed the head of
# the orca-math stream (6,000 kept), so we skip well past that. Using
# GSM8K-train instead would be RL on memorised solutions — healthy reward, no
# generalisation, and a failure that looks like the model simply plateauing.
import json, os
from datasets import load_dataset
sys.path.insert(0, "/content/osrt/src")
from osrt.rewards import extract_numeric_answer

OUT, TARGET, SKIP = "/content/grpo_prompts.jsonl", 6000, 60_000
if not os.path.exists(OUT):
    ds = load_dataset("microsoft/orca-math-word-problems-200k",
                      split="train", streaming=True).skip(SKIP)
    kept = []
    for row in ds:
        if len(kept) >= TARGET:
            break
        q = (row.get("question") or "").strip()
        gold = extract_numeric_answer(row.get("answer") or "")
        if q and gold is not None:
            kept.append({"question": q, "answer": str(gold).strip()})
    with open(OUT, "w") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {len(kept)} prompts")
print(OUT, sum(1 for _ in open(OUT)), "prompts")

# NOTE ON DIFFICULTY SCREENING: ~62-73% of unfiltered prompts are "dead" —
# all rollouts fail, so under group-normalised advantage they contribute
# EXACTLY zero gradient. Screening them out makes each wave ~2.6x more useful,
# but measured at ~13 scans/min it costs hours. Dead prompts are wasteful, not
# harmful, and generation is only a fraction of a step — so we train unfiltered
# and spend the time on more STEPS instead. Revisit if steps get cheap.

In [ ]:
# ── 3. Sanity: 3 steps, tiny wave. Run this BEFORE the long launch. ──
# Checks what no config review can: that the batched loop actually trains,
# VRAM fits, and — most importantly — that the rollouts look sane. The single
# most valuable thing here is READING THE GENERATIONS: a missing system prompt
# once made the model skip <|think|> entirely and eat a -0.5 ambiguity penalty
# on every rollout, which no memory-or-timing check would have caught.
!cd /content/osrt && python scripts/colab_grpo.py \
    --ckpt-dir /content/ckpt --prompts /content/grpo_prompts.jsonl \
    --tokenizer /content/osrt/v6_tokenizer_export \
    --hf-repo HallD/osrt-v6-ckpt \
    --total-steps 3 --num-prompts 4 --ckpt-interval 999 --no-wandb

### Read the sanity output before going further

**Rollouts must look like** `<|think|>…working…<|/think|><|answer|>72<|/answer|>` — think block present, a **single bare number** in the answer block, clean stop.

**Red flags:**
- Completions starting `<|answer|>` with no think block → the system prompt isn't reaching the model.
- Several numbers inside `<|answer|>` → strict extraction rules them `ambiguous` and applies −0.5; reward will sit negative.
- `live 0/N` → every rollout in every group scored identically, so all advantages are zero and **nothing is learning**. That is the recorded collapse mode ("uniform rewards → zero advantage → frozen updates"). Raise `group_size` or check the reward path.
- Reward strongly negative at step 0 → something is wrong; step 0 should be mildly positive since format alone is worth up to +3.0.

Also note **seconds/step** and **VRAM** — they set the wave size and total steps below.

In [ ]:
# ── 4. THE RUN — detached, survives the notebook tab ─────────────────
# nohup + a log file, NOT a foreground cell: Colab sessions drop, and a
# foreground run dies with the websocket. Checkpoints push to HF every 50
# steps so a reclaimed VM costs at most 50 steps; re-running this cell resumes
# from the newest local (or HF) checkpoint automatically.
boot = r'''#!/bin/bash
cd /content/osrt
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python scripts/colab_grpo.py \
  --ckpt-dir /content/ckpt \
  --prompts /content/grpo_prompts.jsonl \
  --tokenizer /content/osrt/v6_tokenizer_export \
  --hf-repo HallD/osrt-v6-ckpt \
  --ckpt-interval 50 \
  --num-prompts 16 \
  --micro-batch 8 \
  --max-gen-len 768 \
  --compile
'''
open("/content/boot.sh", "w").write(boot)
!chmod +x /content/boot.sh
!nohup /content/boot.sh > /content/grpo.log 2>&1 &
print("launched detached -> /content/grpo.log")

In [ ]:
# ── 5. Watch it (re-run this cell whenever) ──────────────────────────
# Steps, rollouts and errors. The rollout lines are the vibe check.
!grep -E "^step|rollouts @|^  \[|saved|pushed|Error|Traceback|CUDA out of memory" \
    /content/grpo.log | tail -40

## Judging the run

**Reward EMA is not the judge.** It rises early for free — the exact-format term alone is +3.0, and the model already scores ~100% on format — so an early jump is *format consolidation*, not reasoning. This project has had loss and accuracy dissociate three separate times.

**What to judge on:**
- **`acc`** in the step line — correctness on the training prompts.
- **`live N/M`** — rollouts with non-zero advantage. If this collapses toward 0, learning has stopped regardless of what reward does.
- **The printed rollouts** — is the reasoning actually working the problem, or reciting its shape? At the start the model invents numbers and botches arithmetic while keeping perfect format; watch whether that changes.
- **Held-out GSM8K** — score the pushed checkpoints with `app.py::sft_eval_sweep` (n≥200; smaller samples cannot resolve a 5pp difference). Baseline to beat: **17.5% acc_on / 12.0% acc_off**.

**Expect a slow grind.** 8B models reportedly need 300+ steps before answers improve; a 601M base needs at least that. Don't read a flat stretch at step 100 as failure — or an early reward spike as success.

**Sweep checkpoints at the end, don't ship the last one.** In SFT-v4 the final checkpoint was measurably *not* the best: accuracy peaked at step 1,800 and the reasoning-on advantage eroded from +7.0pp to +1.0pp as training extended.